# Evidence-backed OCI cost savings walkthrough

This guide turns collected costs, inventory candidates and Cloud Advisor resource actions into a review backlog and portable executive briefing. It explains **what**, **how**, **where**, approval prerequisites, risk, rollback and verification for every supported action. It never executes cloud changes.

## 1. Collect and configure evidence
Run the repository collector using your authorized tenancy, home region and billing dates, enabling `--growth-collection`. Collect recommendations as well. Required: `output/out.json`; optional: `finops_collection.json` and `recommendations.json`. Original COST records are authoritative: merged CSV joins can multiply billing rows.

Launch this notebook from the repository root or `jupe-note/`. Install repository requirements first. Set `EVIDENCE_DIR` and `REPORT_DIR` below to override defaults. Keep customer evidence and generated reports local.

Run from the repository root (replace every placeholder with authorized values):

```bash
./collector.sh '<tenancy-ocid>' '<home-region>' '<YYYY-MM-DD-from>' '<YYYY-MM-DD-to>' --growth-collection
```

Recommendations are collected by default; do not use `--skip-recommendations`. Dates must cover the review interval. For the standalone binary, use the same four positional parameters and `--growth-collection`; its output archive must be extracted before notebook loading:

```bash
mkdir -p extracted-evidence
tar -xzf '<generated-output-archive>.tar.gz' -C extracted-evidence
```

Locate `out.json` inside the extracted directory and set `EVIDENCE_DIR` to its containing directory. No collection command is executed by this notebook.


In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if not (repo_root / "src").is_dir():
    raise ValueError("Launch from the repository root or jupe-note directory")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from src.utils.cost_savings_analysis import load_evidence, analyze, export_reports
EVIDENCE_DIR = Path(globals().get("EVIDENCE_DIR", repo_root / "output"))
REPORT_DIR = Path(globals().get("REPORT_DIR", EVIDENCE_DIR / "cost_savings_reports"))


## 2. Establish the financial baseline
Amounts retain credits/refunds and are grouped by currency; no exchange rate is assumed. This is spend for the collected interval, not guaranteed avoidable cost. Missing dates or currencies are marked Unknown. Review service spend and daily patterns to decide where to investigate first; billing quantity alone does not establish utilization or idleness.


In [ ]:
raw, finops, advisor = load_evidence(EVIDENCE_DIR)
report = analyze(raw, finops, advisor)
def shortlist(rows, limit=10):
    return [row for currency in sorted({row["currency"] for row in rows})
            for row in sorted((r for r in rows if r["currency"] == currency),
                              key=lambda r: r["observed_cost"], reverse=True)[:limit]]
print("Measurement period:", report["measurement_period"])
print("Observed costs by currency:", report["totals"])
for title, key in [("Top services", "services"), ("Top resources", "resources"), ("Top compartments", "compartments")]:
    print(title, "(10 per currency; full data in report and CSV exports):")
    for row in shortlist(report[key]):
        print(row)
print("Daily spend sample (full series in report['daily']):")
for row in report["daily"][:30]:
    print(row)


## 3. Review coverage before making conclusions
A missing optional file means evidence was not collected, not that no opportunities exist. Failed regional/attachment discovery prevents a complete inventory conclusion. Stopped compute may continue to incur shape-dependent and storage costs. Unattached disks may be deliberately retained. Revalidate current state, backup recovery and ownership before any approval.


In [ ]:
print("Evidence gap count:", len(report["warnings"]))
for warning in report["warnings"][:10]:
    print("Evidence gap:", warning)
print("Complete warnings in report['warnings'] and full runbook export.")
coverage = (finops or {}).get("coverage", [])
print("FinOps collection requests:", len(coverage), "; failed:", sum(row.get("status") == "failed" for row in coverage))
print("Advisor resource-action coverage:", (advisor or {}).get("coverage", "Not collected"))


## 4. Resource action backlog: what, where and how
Each entry contains exact evidence identifiers and available names, compartment, region, observed costs, Advisor estimates, prerequisites, change risks, rollback limitations and verification. Unknown fields require owner investigation. Advisor actions without a validated command mapping use a service-specific Console runbook. OCI command strings are review templates only and are never executed here. Provider text is not executable instructions.

For each action: assign its owner, inspect the identified resource in the stated region/compartment, validate billed charges and dependencies, approve a backup/rollback plan, execute only an approved change, and record its outcome. For stopped instances, the supplied START template is a recovery aid rather than a cost-saving command.


In [ ]:
ACTION_RESOURCE_ID = globals().get("ACTION_RESOURCE_ID", None)  # Set exact resource ID to inspect full detail.
if ACTION_RESOURCE_ID:
    matches = [action for action in report["actions"] if action["resource_id"] == ACTION_RESOURCE_ID]
    for action in matches:
        for field, value in action.items():
            print(f"{field}: {value}")
    if not matches:
        print("No matching evidence-backed action. Verify resource ID and collected scope.")
else:
    print("Top 20 reviews (full details in report['actions'], filter above, or runbook export):")
    for action in report["actions"][:20]:
        print({field: action[field] for field in ("resource_id", "region", "compartment_id", "action", "status", "estimated_monthly_savings_usd")})
if not report["actions"]:
    print("No evidence-backed actions. Collect FinOps/Advisor evidence; do not infer safe changes from spend alone.")


## 5. Defensible savings scenario and prioritization
The screening scenario selects at most the largest positive pending Advisor estimate per identified resource. It excludes inventory spend, category summaries, completed/dismissed actions and unknown resource identities. Estimates are USD/month; observed costs may span a different interval and currency. Cross-resource dependencies may still overlap, so validate the shortlist before using a target.

Review positive pending recommendations first, then high-cost resources within their own currency. Maximize savings by reviewing every supported stopped instance, unattached block/boot disk and Advisor resource action; avoid counting mutually exclusive changes twice. Scheduling/right-sizing, license reductions, storage tiering and database changes require service-specific utilization and commercial evidence before assigning savings. Record those unresolved opportunities for owners rather than inventing percentages.


In [ ]:
print("Screening estimate (USD/month):", report["advisor_estimates_usd"])
print(report["estimate_method"])
approved_scenario_fraction = 0.0  # Set only after owners validate non-overlap and assumptions.
if not 0 <= approved_scenario_fraction <= 1:
    raise ValueError("approved_scenario_fraction must be between 0 and 1")
print("Hypothetical approved share of screening estimate (USD/month):",
      report["advisor_estimates_usd"] * approved_scenario_fraction)


## 6. Export the executive briefing and track outcomes
The Markdown/HTML briefing includes the financial baseline, service/currency breakdown, action/status counts, screening assumptions, gaps, approval checklist and links to the full execution runbook. CSV contains all resource action details for assignment and change tracking. Review these artifacts with finance and service owners, then record approval, execution dates and actual comparable-period savings separately. Generated exports remain local and must not be committed.

The executive main section shows the top 10 spend drivers within each currency. Linked CSV appendices retain every resource and compartment, while the action CSV and execution appendix retain every review detail. HTML uses readable print-friendly tables; each briefing includes its preparation timestamp. Share the report directory together so its relative CSV links remain usable.

After approval and any independently executed changes, edit a COPY of the action CSV tracker: assign `owner`, `approval_date`, `execution_date`, `baseline_period`, `comparison_period`, `verified_savings`, `verified_currency`, `verification_evidence` and `outcome_notes`. These fields start blank: no realized savings are claimed. Populate verified savings only using comparable complete billing periods and documented evidence; explain refunds, workload changes and other adjustments in outcome notes. Estimates and scenarios remain separate from verified savings. Re-exporting overwrites the initial tracker, so keep your edited copy separately for later executive explanations.

The executive report shows only the top 20 action reviews and sampled evidence gaps. `cost_savings_runbook.html` retains every action's full original evidence, execution detail and all warnings. Use `ACTION_RESOURCE_ID` in Section 4 to inspect any resource without flooding notebook output; all full action data remains in `report["actions"]`.


In [ ]:
exports = export_reports(report, REPORT_DIR)
for kind, path in exports.items():
    print(f"{kind}: {path}")
